In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/FMCG_2022_2024.csv')

In [ ]:
df

,date,sku,brand,segment,category,channel,region,pack_type,price_unit,promotion_flag,delivery_days,stock_available,delivered_qty,units_sold
0,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-Central,Multipack,2.38,0,1,141,128,9
1,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-North,Single,1.55,1,3,0,129,0
2,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Retail,PL-South,Carton,4.00,0,5,118,161,8
3,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-Central,Single,5.16,0,2,81,114,7
4,2022-01-21,MI-006,MiBrand1,Milk-Seg3,Milk,Discount,PL-North,Single,7.66,0,4,148,204,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190752,2024-12-31,SN-030,SnBrand2,SnackBar-Seg1,SnackBar,Discount,PL-North,Single,2.55,0,2,190,163,25
190753,2024-12-31,SN-030,SnBrand2,SnackBar-Seg1,SnackBar,Discount,PL-South,Single,6.01,0,5,141,131,19
190754,2024-12-31,SN-030,SnBrand2,SnackBar-Seg1,SnackBar,E-commerce,PL-Central,Single,3.45,0,5,0,132,0
190755,2024-12-31,SN-030,SnBrand2,SnackBar-Seg1,SnackBar,E-commerce,PL-North,Multipack,1.93,1,2,211,201,40


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/FMCG_2022_2024.csv')
df['date'] = pd.to_datetime(df['date'])

monthly_df = df.groupby([pd.Grouper(key='date', freq='MS'), 'sku'])['units_sold'].sum().reset_index()
monthly_df.rename(columns={'date': 'Date', 'units_sold': 'Sales'}, inplace=True)
monthly_df = monthly_df.sort_values(['sku', 'Date']).reset_index(drop=True)

le = LabelEncoder()
monthly_df['sku_encoded'] = le.fit_transform(monthly_df['sku'])

In [ ]:
def create_features(data):
    df_feat = data.copy()
    df_feat['Month'] = df_feat['Date'].dt.month
    df_feat['Year'] = df_feat['Date'].dt.year
    df_feat['Quarter'] = df_feat['Date'].dt.quarter

    df_feat['Lag_1'] = df_feat.groupby('sku')['Sales'].shift(1)
    df_feat['Lag_2'] = df_feat.groupby('sku')['Sales'].shift(2)
    df_feat['Lag_3'] = df_feat.groupby('sku')['Sales'].shift(3)
    df_feat['Lag_6'] = df_feat.groupby('sku')['Sales'].shift(6)
    return df_feat.dropna().reset_index(drop=True)

df_model = create_features(monthly_df)

In [ ]:
split_date = df_model['Date'].max() - pd.DateOffset(months=5)
train = df_model[df_model['Date'] < split_date].copy()
test = df_model[df_model['Date'] >= split_date].copy()

FEATURES = ['sku_encoded', 'Month', 'Year', 'Quarter', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_6']
TARGET = 'Sales'

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [ ]:
print("Melatih XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

print("Melatih LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, min_child_samples=2, verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_preds = lgb_model.predict(X_test)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_preds))

print("-" * 30)
print(f"RMSE XGBoost  : {xgb_rmse:.2f}")
print(f"RMSE LightGBM : {lgb_rmse:.2f}")
print("-" * 30)

# Logika Pemilihan Otomatis
if xgb_rmse < lgb_rmse:
    print("🏆 KEPUTUSAN: XGBoost memiliki error lebih kecil. Menggunakan XGBoost!")
    best_preds = xgb_preds
else:
    print("🏆 KEPUTUSAN: LightGBM memiliki error lebih kecil. Menggunakan LightGBM!")
    best_preds = lgb_preds

Melatih XGBoost...
Melatih LightGBM...
------------------------------
RMSE XGBoost  : 310.76
RMSE LightGBM : 316.25
------------------------------
🏆 KEPUTUSAN: XGBoost memiliki error lebih kecil. Menggunakan XGBoost!


In [ ]:
test['Forecast'] = best_preds.round(0)

# Gabungkan dengan train data
final_df = pd.concat([train, test], axis=0)

export_df = final_df[['Date', 'sku', 'Sales', 'Forecast']]
export_df.to_csv('FMCG_SKU_Forecast_Best.csv', index=False)

print("\n[BERHASIL] File 'FMCG_SKU_Forecast_Best.csv' siap diunduh untuk Streamlit!")


[BERHASIL] File 'FMCG_SKU_Forecast_Best.csv' siap diunduh untuk Streamlit!
